In [1]:
import os
import time
import numpy as np
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
llm = ChatOpenAI(model='gpt-4o-mini')
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [ ]:
# RAG를 실험적인 코드로 하는 것

In [4]:
from pathlib import Path

data_dir = Path('sample_data')

files = {
    data_dir / 'company_policy.txt' : '사내규정',
    data_dir / 'product_manual.txt' : '제품메뉴얼',
    data_dir / 'ai_report.txt' : 'AI 보고서',
}

documents = []
for fpath, category in files.items():
    text = fpath.read_text()
    for section in text.strip().split('\n\n'):
        if section.strip():
            documents.append(Document(
                page_content = section.strip(), 
                metadata = {
                          "category" : category,
                          "source" : fpath.name
                         }))

In [5]:
len(documents)

13

In [6]:
documents[0]

Document(metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='주식회사 모두의연구소 사내 규정')

In [7]:
vectorstore = FAISS.from_documents(documents, embeddings)

In [8]:
retriever = vectorstore.as_retriever(search_kwarg={'k':3})

In [9]:
# 임베딩에 대한 값은 저장하면 좋음
vectorstore.save_local('faiss_docs')

In [10]:
loaded_vs = FAISS.load_local('faiss_docs', embeddings, allow_dangerous_deserialization=True)

In [12]:
loaded_vs.index.ntotal

13

In [13]:
retriever.invoke('재택근무 규정')

[Document(id='c804386e-4eef-4cee-8ec9-72d0547bf927', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제2조 (근무시간)\n기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n재택근무는 주 2회까지 가능하다.'),
 Document(id='7298cee4-b679-44e9-9fa0-07781bc540af', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제3조 (휴가)\n연차휴가는 근로기준법에 따라 부여한다.\n경조사 휴가는 별도 규정에 따른다.\n자기개발 휴가를 연 5일 추가 부여한다.'),
 Document(id='f68a6238-f338-46c8-a45f-fdd86893ef65', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.'),
 Document(id='0b5da37e-dad1-484d-9d65-f926df962a85', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제4조 (교육)\n모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n온라인 학습 플랫폼 이용료를 전액 지원한다.')]

In [ ]:
# LCEL : Langchain 문법 | 파이프로 연결해서 진행하는 것 Langchain Expression Language

In [14]:
simple_prompt = ChatPromptTemplate.from_messages([
    ('user', '{question}')
])

simple_chain = simple_prompt | llm | StrOutputParser()

In [15]:
simple_chain.invoke({'question' : '대한민국의 수도는?'})

'대한민국의 수도는 서울입니다.'

In [ ]:
# retriever 조회하다
# RAG는 query를 입력 -> retriever 검색을 처리 -> llm : 사용자의 question(query), retriever documents

In [16]:
# RunnablePassthrough : 입력된 데이터를 그대로 반환
from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough()

In [17]:
# invoke 부르다, = call
passthrough.invoke('안녕하세요')

'안녕하세요'

In [18]:
passthrough.invoke(123)

123

In [20]:
def format_docs(docs):
    return '\n-\n'.join(f"[{d.metadata.get('category', '')}] {d.page_content}" for d in docs)

In [21]:
docs = retriever.invoke('재택근무')

In [22]:
formatted = format_docs(docs)

In [23]:
print(formatted)

[사내규정] 제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.
-
[사내규정] 제4조 (교육)
모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
온라인 학습 플랫폼 이용료를 전액 지원한다.
-
[사내규정] 제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.
-
[사내규정] 제3조 (휴가)
연차휴가는 근로기준법에 따라 부여한다.
경조사 휴가는 별도 규정에 따른다.
자기개발 휴가를 연 5일 추가 부여한다.


In [24]:
retriever_chain = retriever | format_docs
context_text = retriever_chain.invoke('재택근무')

In [25]:
print(context_text)

[사내규정] 제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.
-
[사내규정] 제4조 (교육)
모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
온라인 학습 플랫폼 이용료를 전액 지원한다.
-
[사내규정] 제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.
-
[사내규정] 제3조 (휴가)
연차휴가는 근로기준법에 따라 부여한다.
경조사 휴가는 별도 규정에 따른다.
자기개발 휴가를 연 5일 추가 부여한다.


In [40]:
rag_prompt = ChatPromptTemplate.from_messages([
    ('system', '사내 도우미 챗봇입니다. 참고문서 :\n{context}\n\n문서 기반으로 답변해주세요.'),
    ('user', '{question}')
])

rag_chain = ({'context': retriever | format_docs, 'question' : RunnablePassthrough()}
             | rag_prompt
             | llm
             | StrOutputParser()
            )

In [30]:
answer = rag_chain.invoke('재택근무 규정이 어떻게 되나요?')
print(answer)

재택근무는 주  2회까지 가능하다고 규정되어 있습니다.


In [31]:
question_list = ['스마트홈 허브 초기 설정 방법', 'ai 산업 성장률', '교육비 지원 한도']

for q in question_list:
    print(f'Q: {q}')
    print(f'A: {rag_chain.invoke(q)}')
    print('======')

Q: 스마트홈 허브 초기 설정 방법
A: 스마트 홈 허브 v3.0의 초기 설정 방법은 다음과 같습니다:

1. **전원 연결**: 스마트 홈 허브의 전원을 연결합니다.
2. **Wi-Fi 네트워크 접속**: 허브가 Wi-Fi 네트워크에 접속하도록 설정합니다.
3. **모바일 앱 설치**: 스마트폰에 모바일 앱을 설치하고 화면에 표시된 QR 코드를 스캔합니다.
4. **IoT 기기 등록**: 연동할 IoT 기기를 검색하고 등록하여 설정을 완료합니다.

이 과정을 따라 스마트 홈 허브를 쉽게 설정할 수 있습니다.
Q: ai 산업 성장률
A: 2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했습니다. 이 중 특히 생성형 AI 분야가 전체 성장의 60%를 견인한 것으로 나타났습니다.
Q: 교육비 지원 한도
A: 사내 규정에 따르면 외부 컨퍼런스 참석비는 연 200만원까지 지원됩니다. 또한 온라인 학습 플랫폼 이용료는 전액 지원됩니다.


In [32]:
from langchain_core.runnables import RunnableParallel

In [33]:
rag_chain_with_sources = RunnableParallel(
    answer = rag_chain,
    source_document = retriever
)

In [34]:
rag_chain_with_sources.invoke('재택근무 규정이 어떻게 되나요?')

{'answer': '재택근무는 주  2회까지 가능합니다.',
 'source_document': [Document(id='c804386e-4eef-4cee-8ec9-72d0547bf927', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제2조 (근무시간)\n기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n재택근무는 주 2회까지 가능하다.'),
  Document(id='7298cee4-b679-44e9-9fa0-07781bc540af', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제3조 (휴가)\n연차휴가는 근로기준법에 따라 부여한다.\n경조사 휴가는 별도 규정에 따른다.\n자기개발 휴가를 연 5일 추가 부여한다.'),
  Document(id='f68a6238-f338-46c8-a45f-fdd86893ef65', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.'),
  Document(id='0b5da37e-dad1-484d-9d65-f926df962a85', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='제4조 (교육)\n모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n온라인 학습 플랫폼 이용료를 전액 지원한다.')]}

In [36]:
result = rag_chain_with_sources.invoke('재택근무 규정이 어떻게 되나요?')
for doc in result['source_document']:
    print(f"[{doc.metadata['category']}] {doc.page_content[:100]}")

[사내규정] 제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.
[사내규정] 제3조 (휴가)
연차휴가는 근로기준법에 따라 부여한다.
경조사 휴가는 별도 규정에 따른다.
자기개발 휴가를 연 5일 추가 부여한다.
[사내규정] 제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.
[사내규정] 제4조 (교육)
모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
온라인 학습 플랫폼 이용료를 전액 지원한다.


In [39]:
rag_chain.invoke('2022년 월드컵 우승컵을 누가 했는지 알려주세요')

'죄송하지만, 제공된 문서에는 2022년 월드컵 우승컵에 대한 정보가 포함되어 있지 않습니다. 다른 질문이나 문서 관련 정보를 요청하시면 도와드리겠습니다.'

In [41]:
from langchain_core.runnables import RunnableBranch, RunnableLambda

In [47]:
def check_and_prepare(question):
    results = vectorstore.similarity_search_with_score(question, k=3)
    docs = [doc for doc, _ in results]
    return {
        'question' : question,
        'context' : format_docs(docs),
        'score' : results[0][1]
    }

# 벡터간의 거리를 가지고 score 처리 x['score'] 거리가 먼 애들은 해당 정보가 없습니다.
# 거리가 가까운 것은 llm으로 넘어감
safe_chain = (
    RunnableLambda(check_and_prepare)
    | RunnableBranch(
        (lambda x: x['score'] > 1.3, lambda x : '해당 정보가 없습니다.'),
        rag_prompt | llm | StrOutputParser()
))

safe_chain.invoke('2002년 월드컵 우승팀은 어디인가요?')

'해당 정보가 없습니다.'

In [49]:
# RunnableParallel : answer, source_documents 검색된 텍스트 context를 동시에 리턴하는 체인
rag_chain_with_sources = RunnableParallel(
    answer=rag_chain, source_document=retriever, context=retriever_chain,
)

rag_chain_with_sources.invoke("2002년 월드컵 우승팀은 어디인가요?")

{'answer': '죄송하지만, 제가 가진 정보는 2023년 10월까지의 데이터로 제한되어 있어, 2002년 월드컵 우승팀에 대한 정보는 포함되어 있지 않습니다. 하지만, 2002년 FIFA 월드컵의 우승팀은 브라질입니다. 추가로 필요한 정보가 있으시면 말씀해 주세요!',
 'source_document': [Document(id='39e1a55b-7392-437c-8330-d17fabfd5d7b', metadata={'category': 'AI 보고서', 'source': 'ai_report.txt'}, page_content='2024년 인공지능 산업 동향 보고서'),
  Document(id='2213afd3-fbed-46ab-bd60-2a3942180d6a', metadata={'category': 'AI 보고서', 'source': 'ai_report.txt'}, page_content='개요\n2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.\n특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.'),
  Document(id='61740988-6d53-4f59-826a-aaf6a9ee631a', metadata={'category': 'AI 보고서', 'source': 'ai_report.txt'}, page_content='시장 전망\n2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.\n특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.'),
  Document(id='0d532a25-9cd4-40d8-9a42-fb184dfbcc0e', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='주식회사 모두의연구소 사내 규정')],
 'context': '[AI 보고서] 2024년 인공지능 산업 동향 보고서\n-\n[AI 보고서] 개요\n2024년 인공지능 산업은 전년 대비

In [50]:
retriever.invoke('2002년 월드컵')

[Document(id='39e1a55b-7392-437c-8330-d17fabfd5d7b', metadata={'category': 'AI 보고서', 'source': 'ai_report.txt'}, page_content='2024년 인공지능 산업 동향 보고서'),
 Document(id='2213afd3-fbed-46ab-bd60-2a3942180d6a', metadata={'category': 'AI 보고서', 'source': 'ai_report.txt'}, page_content='개요\n2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.\n특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.'),
 Document(id='61740988-6d53-4f59-826a-aaf6a9ee631a', metadata={'category': 'AI 보고서', 'source': 'ai_report.txt'}, page_content='시장 전망\n2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.\n특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.'),
 Document(id='0d532a25-9cd4-40d8-9a42-fb184dfbcc0e', metadata={'category': '사내규정', 'source': 'company_policy.txt'}, page_content='주식회사 모두의연구소 사내 규정')]

In [52]:
for doc, score in vectorstore.similarity_search_with_score('재택 근무 규정', k=3):
    print(f"[{doc.metadata['category']}] {doc.page_content}")

[사내규정] 제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.
[사내규정] 제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.
[사내규정] 제3조 (휴가)
연차휴가는 근로기준법에 따라 부여한다.
경조사 휴가는 별도 규정에 따른다.
자기개발 휴가를 연 5일 추가 부여한다.


In [54]:
test_queries = [
    {'query' : '재택근무 몇 회?'},
    {'query' : '스마트홈 초기 설정'},
    {'query' : 'RAG 기술 동햐'},
]

for tq in test_queries:
    results = vectorstore.similarity_search_with_score(tq['query'], k=1)
    top_cat = results[0][0].metadata['category']
    top_score = results[0][1]
    print(f"{tq}: top caterory : {top_cat}, {top_score}")

{'query': '재택근무 몇 회?'}: top caterory : 사내규정, 0.97711181640625
{'query': '스마트홈 초기 설정'}: top caterory : 제품메뉴얼, 0.8594766855239868
{'query': 'RAG 기술 동햐'}: top caterory : AI 보고서, 1.1561551094055176


In [56]:
def verify_query(query, threshold=1.3, k=3):
    results = vectorstore.similarity_search_with_score(query, k=k)

    best_doc = None
    best_score = float('inf')

    for doc, score in results:
        if score < best_score:
            best_score = score
            best_doc = doc

    if best_score > threshold:
        return '해당 정보가 없습니다.'

    return best_doc.page_content

In [57]:
verify_query('2002년 월드컵')

'해당 정보가 없습니다.'